In [5]:
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import timedelta
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                             accuracy_score, precision_recall_fscore_support,
                             roc_auc_score, confusion_matrix, RocCurveDisplay)
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt
import seaborn as sns

ModuleNotFoundError: No module named 'sklearn'

In [1]:
# -----------------------
# Config projet
# -----------------------
LOGS_PATH = "logs_info_25_pseudo.csv"  # logs ARCHE
NOTES_PATH = "notes.csv"               # pseudo,note
COURSE_FILTER = None                   # ex: "PASS - S1 - UE ..." (None = tous)
SUCCESS_THRESHOLD = 10.0               # note >= 10 => réussite
SESSION_GAP_MIN = 30                   # minutes: seuil de nouvelle session
MAX_GAP_MIN = 15                       # pour approx temps passé (cap des deltas)

rng = np.random.RandomState(42)


NameError: name 'np' is not defined

In [3]:

# -----------------------
# ETL : lecture / nettoyage minimal
# -----------------------
def load_logs(path=LOGS_PATH, course_filter=COURSE_FILTER):
    df = pd.read_csv(path)
    # Normalisation colonnes attendues
    expected = {"heure","pseudo","contexte","composant","evenement"}
    missing = expected - set(df.columns.str.lower())
    # Harmoniser le nom exact (en minuscules) si besoin
    df.columns = [c.lower() for c in df.columns]
    assert set(df.columns) >= expected, f"Colonnes manquantes: {missing}"
    # Parse datetime
    df["heure"] = pd.to_datetime(df["heure"], errors="coerce")
    df = df.dropna(subset=["heure","pseudo"])
    # Option : filtrer un cours spécifique (si tu veux modéliser par cours)
    if course_filter:
        df = df[df["contexte"].astype(str).str.contains(course_filter, na=False)]
    # Cours ID (extraction très simple : avant la virgule / ou pattern)
    df["course_id"] = df["contexte"].astype(str).str.extract(r"^([^,]+)")
    df["pseudo"] = df["pseudo"].astype(str)
    return df

def load_notes(path=NOTES_PATH):
    notes = pd.read_csv(path)
    # colonnes attendues : pseudo, note
    notes.columns = [c.lower() for c in notes.columns]
    assert {"pseudo","note"} <= set(notes.columns), "notes.csv doit contenir 'pseudo','note'"
    notes["pseudo"] = notes["pseudo"].astype(str)
    # nettoyages
    notes = notes.dropna(subset=["pseudo","note"])
    return notes


In [4]:
# -----------------------
# Feature engineering
# -----------------------
def _compute_time_spent(group):
    """Approximer temps passé (en minutes) par somme des min(delta_t, MAX_GAP_MIN)."""
    times = group.sort_values("heure")["heure"].values
    if len(times) < 2:
        return 0.0
    deltas = (pd.Series(times[1:]) - pd.Series(times[:-1])).dt.total_seconds() / 60.0
    deltas = np.clip(deltas, 0, MAX_GAP_MIN)
    return float(deltas.sum())

def _compute_sessions(group):
    """Nombre de sessions en considérant un gap >= SESSION_GAP_MIN comme nouvelle session."""
    ts = group.sort_values("heure")["heure"]
    gaps = ts.diff().dt.total_seconds().div(60.0).fillna(0)
    return int((gaps >= SESSION_GAP_MIN).sum() + 1)

def build_features(logs: pd.DataFrame) -> pd.DataFrame:
    df = logs.copy()
    df["date"] = df["heure"].dt.date
    df["hour"] = df["heure"].dt.hour
    df["dow"] = df["heure"].dt.dayofweek  # 0=lundi
    df["is_night"] = df["hour"].between(22, 23) | df["hour"].between(0, 6)
    df["is_weekend"] = df["dow"].isin([5,6])

    # Agrégations par (course_id, pseudo)
    agg = df.groupby(["course_id","pseudo"]).agg(
        total_events=("evenement","count"),
        days_active=("date","nunique"),
        n_forum=("composant", lambda s: (s=="Forum").sum()),
        n_test=("composant",  lambda s: (s=="Test").sum()),
        n_file=("composant",  lambda s: (s=="Fichier").sum()),
        n_course=("composant", lambda s: (s=="Système").sum()),
        n_other=("composant",  lambda s: (~s.isin(["Forum","Test","Fichier","Système"])).sum()),
        n_contexts=("contexte","nunique"),
        night_events=("is_night","sum"),
        wknd_events=("is_weekend","sum"),
    ).reset_index()

    # Sessions & temps passé (fonction sur chaque groupe)
    sessions = df.groupby(["course_id","pseudo"]).apply(_compute_sessions).rename("n_sessions")
    tspent = df.groupby(["course_id","pseudo"]).apply(_compute_time_spent).rename("time_spent_min")
    agg = agg.merge(sessions, on=["course_id","pseudo"]).merge(tspent, on=["course_id","pseudo"])

    # Ratios
    agg["ratio_night"] = agg["night_events"] / agg["total_events"].clip(lower=1)
    agg["ratio_wkend"] = agg["wknd_events"] / agg["total_events"].clip(lower=1)
    agg["events_per_day"] = agg["total_events"] / agg["days_active"].clip(lower=1)
    agg["events_per_session"] = agg["total_events"] / agg["n_sessions"].clip(lower=1)
    return agg


NameError: name 'pd' is not defined

In [ ]:

# -----------------------
# Merge avec les notes + définition du label
# -----------------------
def assemble_dataset(features: pd.DataFrame, notes: pd.DataFrame, threshold=SUCCESS_THRESHOLD):
    # Si plusieurs cours : joindre par (course_id, pseudo) si les notes sont par cours
    # Sinon, si les notes n’ont que pseudo : joindre sur pseudo.
    if {"course_id","pseudo"} <= set(notes.columns):
        data = features.merge(notes, on=["course_id","pseudo"], how="inner")
    else:
        data = features.merge(notes, on="pseudo", how="inner")

    # Label binaire
    data["success"] = (data["note"] >= threshold).astype(int)
    return data

# -----------------------
# Évaluation utilitaires
# -----------------------
def evaluate_regression(y_true, y_pred, name="Regression"):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred, squared=False)
    r2 = r2_score(y_true, y_pred)
    print(f"[{name}] MAE={mae:.3f} | RMSE={rmse:.3f} | R²={r2:.3f}")
    return {"MAE":mae, "RMSE":rmse, "R2":r2}

def evaluate_classification(y_true, y_prob, threshold=0.5, name="Classifier"):
    y_pred = (y_prob >= threshold).astype(int)
    acc = accuracy_score(y_true, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
    try:
        auc = roc_auc_score(y_true, y_prob)
    except Exception:
        auc = np.nan
    print(f"[{name}] ACC={acc:.3f} | P={prec:.3f} | R={rec:.3f} | F1={f1:.3f} | AUC={auc:.3f}")
    print("Confusion matrix:\n", confusion_matrix(y_true, y_pred))
    return {"ACC":acc,"P":prec,"R":rec,"F1":f1,"AUC":auc}

# -----------------------
# Main: ETL -> Features -> Modèles -> Comparaison
# -----------------------
def main():
    # 1) ETL
    logs = load_logs()
    notes = load_notes()

    # 2) Features
    feats = build_features(logs)

    # 3) Dataset final (X num) + y (note) + y_bin (success)
    data = assemble_dataset(feats, notes, threshold=SUCCESS_THRESHOLD)

    # Variables d'entrée = toutes les features numeriques
    id_cols = ["course_id","pseudo"]
    y_reg = data["note"].values
    y_bin = data["success"].values

    X = data.drop(columns=id_cols + ["note","success"])

    # 4) Split train/test (stratifié sur réussite pour évaluation honnête)
    X_train, X_test, yreg_train, yreg_test, ybin_train, ybin_test = train_test_split(
        X, y_reg, y_bin, test_size=0.2, random_state=42, stratify=y_bin
    )

    numeric_features = X.columns.tolist()
    # (pas d'encodage ici car on n'a gardé que des features numériques)

    # ------------------ A) Régression multiple (prédire la note)
    # On compare LinearRegression et Ridge (régression multiple régularisée)
    reg_lin = Pipeline(steps=[
        ("scaler", StandardScaler(with_mean=True, with_std=True)),
        ("reg", LinearRegression())
    ])
    reg_ridge = Pipeline(steps=[
        ("scaler", StandardScaler()),
        ("reg", Ridge())
    ])

    reg_ridge_gs = GridSearchCV(
        reg_ridge,
        {"reg__alpha":[0.1, 1.0, 5.0, 10.0]},
        cv=5, scoring="neg_root_mean_squared_error", n_jobs=-1
    )

    # Entraînement
    reg_lin.fit(X_train, yreg_train)
    reg_ridge_gs.fit(X_train, yreg_train)

    # Évaluation régression
    print("\n=== Évaluation Régression (prédiction de la note) ===")
    yreg_hat_lin = reg_lin.predict(X_test)
    evaluate_regression(yreg_test, yreg_hat_lin, name="LinearRegression")

    best_ridge = reg_ridge_gs.best_estimator_
    yreg_hat_ridge = best_ridge.predict(X_test)
    ridge_scores = evaluate_regression(yreg_test, yreg_hat_ridge, name=f"Ridge(alpha={reg_ridge_gs.best_params_['reg__alpha']})")

    # Conversion en réussite/échec avec le même seuil (comparaison cohérente)
    print("\n--- Classification induite (depuis la régression) ---")
    yprob_from_reg = (yreg_hat_ridge - SUCCESS_THRESHOLD) / 20.0 + 0.5  # simple mapping (optionnel)
    yprob_from_reg = np.clip(yprob_from_reg, 0, 1)
    reg_cls_scores = evaluate_classification(ybin_test, yprob_from_reg, name="Reg->Success")

    # ------------------ B) Random Forest (classification directe de la réussite)
    rf = RandomForestClassifier(
        n_estimators=400,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        random_state=42,
        n_jobs=-1
    )
    rf.fit(X_train, ybin_train)
    yprob_rf = rf.predict_proba(X_test)[:,1]
    print("\n=== Évaluation Classification (Random Forest) ===")
    rf_scores = evaluate_classification(ybin_test, yprob_rf, name="RandomForest")

    # Importance des variables (interprétation)
    importances = pd.Series(rf.feature_importances_, index=numeric_features).sort_values(ascending=False)
    print("\nTop 15 features importantes (RF) :")
    print(importances.head(15))

    # Courbe ROC (optionnel)
    try:
        RocCurveDisplay.from_predictions(ybin_test, yprob_rf)
        plt.title("ROC – Random Forest")
        plt.show()
    except Exception:
        pass

    # Bilan court
    print("\n=== BILAN COMPARATIF ===")
    print("Régression (Ridge) sur note :", ridge_scores)
    print("Classification (RF) sur succès :", rf_scores)
    # Décision : tu motiveras le choix par la métrique principale (p.ex. F1/ROC-AUC pour déséquilibre)

if __name__ == "__main__":
    main()